# VLM Fundamentals

**Module:** 15 — VLMs & Multimodal

Vision-language models join pixels and tokens so systems can see, read, and reason over images.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Define vision-language models and multimodal learning precisely
- Explain how images become tokens/features for LLMs
- List core VLM capabilities and hard limitations
- Construct a minimal multimodal API message (OpenAI-style)
- Estimate image-token cost and choose detail/resolution wisely


## What Is a Vision-Language Model?

### Definition
A **VLM** jointly processes visual inputs (images, pages, frames) and text to produce text or structured outputs conditioned on both modalities.

### Why it matters
Receipts, UIs, charts, and scans are multimodal by nature. Text-only LLMs invent what they cannot see.

### How it works
Vision encoder → projector/adapter → LLM decoder attending over interleaved visual + text tokens.

### Intuition
Treat the image as a second document tokenized into visual words.

### Pitfalls
- Assuming human-like vision at any resolution
- Huge unresized images → cost/context blowups
- Treating outputs as perfect OCR

### When to use
Captioning, VQA, DocQA, grounding, multimodal RAG, accessibility, UI agents.


### Capability map

| Capability | Input | Output | Common failure |
|------------|-------|--------|----------------|
| Captioning | Image | Description | Hallucinated objects |
| VQA | Image+Q | Answer | Answering from priors |
| DocQA / OCR | Page | Text/JSON | Wrong column/total |
| Grounding | Phrase | BBox/region | Confident wrong box |
| MM embeddings | Image/text | Vector | Domain shift |

```mermaid
flowchart LR
  I[Image] --> E[Vision encoder]
  E --> P[Projector]
  P --> V[Visual tokens]
  T[Text] --> L[LLM]
  V --> L --> O[Text / JSON / tools]
```


In [ ]:
# Demo 1: OpenAI-style multimodal request + key placeholder
import os, json, base64
from pathlib import Path

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY")

def to_data_url(path: str, mime="image/png") -> str:
    raw = Path(path).read_bytes() if Path(path).exists() else b"fake-image-bytes"
    b64 = base64.b64encode(raw).decode("ascii")
    return f"data:{mime};base64,{b64 if len(b64)<40 else b64[:32]+'...'}"

req = {
    "model": "gpt-4o",
    "messages": [
        {"role": "system", "content": "You are a precise visual analyst."},
        {"role": "user", "content": [
            {"type": "text", "text": "Describe this image in one sentence."},
            {"type": "image_url", "image_url": {"url": "https://example.com/chart.png", "detail": "high"}},
        ]},
    ],
    "max_tokens": 200,
}
print(json.dumps(req, indent=2)[:450], "...")
print("key:", OPENAI_API_KEY[:12] + "...", "data:", to_data_url("x.png")[:50])


In [ ]:
# Demo 2: realistic chat.completion response shape (vision)
import json
resp = {
    "id": "chatcmpl-vlm-demo",
    "object": "chat.completion",
    "model": "gpt-4o",
    "choices": [{"index": 0, "finish_reason": "stop", "message": {
        "role": "assistant",
        "content": "A bar chart of quarterly revenue; Q3 is highest. Y-axis labeled Revenue (USD).",
    }}],
    "usage": {"prompt_tokens": 1420, "completion_tokens": 36, "total_tokens": 1456},
}
print(json.dumps(resp, indent=2))
print("Note: prompt_tokens includes image tokens.")


## Modalities & Representations

### Definition
A **modality** is a data channel (vision, text, audio). Multimodal learning aligns or fuses representations across channels.

### Why it matters
Product bugs often come from EXIF orientation, DPI, PDF raster scale, or RGBA vs RGB mismatches.

### How it works
Images → patches/regions; text → subwords; alignment via contrastive and/or generative training.

### Intuition
Pixels are continuous; language is discrete — the projector translates.

### Pitfalls
- Ignoring EXIF rotation
- JPEG crushing small text
- One video frame pretending to be video understanding

### When to use
Always normalize ingest before the model call.


### Image → tokens pipeline

| Stage | Action | Cost lever |
|-------|--------|------------|
| Decode | Bytes → tensor | Format |
| Resize/tile | Fit resolution | `detail`, tile count |
| Encode | Patches → embeddings | Tower choice |
| Project | Map to LLM width | Adapter |
| Pack | Interleave in context | Budget |

```
[patches] -> Encoder -> [v1..vN] -> Projector -> [t1..tN] \
"What is total?" -------------------------------> LLM -> answer
```


In [ ]:
# Demo 3: educational image-token estimator
from dataclasses import dataclass

@dataclass
class ImageTokenEstimate:
    width: int; height: int; detail: str = "high"; patch: int = 32
    def tokens(self) -> int:
        if self.detail == "low":
            return 85
        tiles = min(max(1,(self.width+511)//512) * max(1,(self.height+511)//512), 4)
        return tiles * (512//self.patch)**2 + 85

for w,h,d in [(512,512,"low"),(1024,768,"high"),(3000,4000,"high")]:
    print(f"{w}x{h} {d} ~ {ImageTokenEstimate(w,h,d).tokens()} tokens")


In [ ]:
# Demo 4: two-image comparison message
import json
msg = {"role": "user", "content": [
    {"type": "text", "text": "Compare chart A vs chart B in 3 bullets. Cite visible axis labels."},
    {"type": "image_url", "image_url": {"url": "https://example.com/a.png", "detail": "high"}},
    {"type": "image_url", "image_url": {"url": "https://example.com/b.png", "detail": "high"}},
]}
print(json.dumps(msg, indent=2))


## Limitations & Design Mitigations

### Definition
VLMs are strong pattern matchers, not perfect sensors — they miss fine print and can invent fluent text.

### Why it matters
High-stakes extraction needs schemas, checksums, and HITL — never 'model said so' alone.

### How it works
Mitigate with crops, higher detail, structured outputs, OCR hybrid, confidence routing.

### Intuition
If a human needs a magnifier, the default thumbnail pass will too.

### Pitfalls
- No provenance/bboxes
- Blind KYC/medical trust
- PII images in logs

### When to use
Bake limitations into architecture on day one.


In [ ]:
# Demo 5: VLM vs classic CV+OCR chooser
def choose_stack(task: str, needs_boxes: bool, strict_audit: bool) -> str:
    t = task.lower()
    if needs_boxes and strict_audit:
        return "detector/OCR + VLM verifier (+ HITL)"
    if any(k in t for k in ("chart", "ui", "open_qa")):
        return "VLM primary"
    if "plain_scan" in t and strict_audit:
        return "classic OCR primary, VLM layout QA"
    return "VLM first; add OCR if field errors high"

for case in [("open_qa screenshot",False,False),("plain_scan invoice",False,True),("receipt boxes",True,True)]:
    print(f"{case[0]:22} -> {choose_stack(*case)}")


In [ ]:
# Demo 6: structured output schema sketch for perception
schema = {
    "name": "scene_parse",
    "schema": {
        "type": "object",
        "required": ["caption", "objects", "readable_text"],
        "properties": {
            "caption": {"type": "string"},
            "objects": {"type": "array", "items": {"type": "object", "properties": {
                "label": {"type": "string"}, "bbox": {"type": "array", "items": {"type": "number"}},
            }}},
            "readable_text": {"type": "array", "items": {"type": "string"}},
            "uncertainty": {"type": "array", "items": {"type": "string"}},
        },
    },
}
print(schema["name"], "keys=", list(schema["schema"]["properties"]))


### Intuition: context window as a lightbox

You can only put so many photos on a light table. Each high-detail image crowds out instructions, tools, and history.
**Budget ritual:** resize → crop ROI → choose `detail` → estimate tokens → then call.


### Checklist — Before first production VLM call

- [ ] Images normalized (orientation, max dimension)
- [ ] API key only from env (`YOUR_OPENAI_API_KEY` placeholder in code)
- [ ] Token estimate logged per request
- [ ] Validator/schema on outputs
- [ ] PII retention policy for raw images


### Try it yourself — Multimodal messages

1. Add `response_format: json_object` to Demo 1 and sketch expected keys.
2. Warn when estimated tokens + 500 text tokens exceed 8000.
3. Write a function that downscales dimensions preserving aspect so max side ≤ 2048.

**Stretch:** Call a live vision API only with env-stored keys; print usage.prompt_tokens.


### Try it yourself — Stack choice

1. For a bank check deposit flow, pick a stack and list 3 validators.
2. List 5 questions for a vendor image-retention policy.


## Knowledge Check

**Q1.** Why can `prompt_tokens` jump when you attach one PNG?

<details><summary>Answer</summary>

The API bills visual tokens derived from resolution/detail/tiling, often dominating text tokens.

</details>

**Q2.** Name two mitigations for hallucinated invoice totals.

<details><summary>Answer</summary>

OCR hybrid reconcile; math checks; HITL on disagreement; schema + confidence routing.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `VLM` | Model jointly handling vision + language |
| `modality` | Data channel such as image or text |
| `projector` | Maps vision embeddings into LLM space |
| `image tokens` | Budgeted visual tokens in the prompt |
| `detail` | API knob trading resolution/tokens vs cost |
| `grounding` | Linking language to spatial regions |
| `HITL` | Human-in-the-loop review |


## Key Takeaways

- VLMs fuse vision features into an LLM so answers condition on pixels
- Multimodal messages interleave text parts and image_url parts
- Image tokens dominate cost — resize, crop, choose detail
- Production needs validators, hybrids, and retention policies


## Production Incident Patterns — VLM fundamentals

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Sudden cost spike | `detail=high` on huge pages | Resize + tile budget |
| Fluent wrong fields | VLM hallucination | OCR hybrid + schema |
| Cross-customer leak | Missing tenant filter | ACL in retriever code |
| Flaky eval scores | Unfrozen prompts/models | Pin versions + bakeoff set |
| Latency SLO burn | Full-page high detail | Crop ROI → mini model |

```
ASCII control loop:
  ingest -> normalize -> route model -> generate -> validate -> (HITL|export)
                     ^                              |
                     +-------- metrics/audit <------+
```


In [ ]:
# Cross-cutting: redact secrets before logging multimodal payloads
import re, json

SECRET_RE = re.compile(r"(api[_-]?key|bearer\s+[A-Za-z0-9._\-]+)", re.I)

def safe_log(payload: dict) -> str:
    s = json.dumps(payload)
    s = SECRET_RE.sub("***", s)
    if "base64," in s:
        s = re.sub(r"base64,[A-Za-z0-9+/=]+", "base64,[REDACTED]", s)
    return s[:500]

print(safe_log({
    "model": "gpt-4o",
    "api_key": "YOUR_OPENAI_API_KEY",
    "content": "data:image/png;base64,AAAABBBBCCCC",
    "topic": "VLM fundamentals",
}))


## Mini Case Study — VLM fundamentals

**Scenario:** A team ships a vision feature in one week. Demo looks great on three happy-path images.
**Week 2:** finance reports wrong totals; legal asks about image retention; GPU/API bill 4× forecast.

**Retro questions**
1. What was the output contract (schema) on day one?
2. Which failure mode had no metric?
3. Was there a crop/detail budget?
4. Who owns HITL and appeals?

**Design rule:** if a field can move money or identity, it needs a validator + disagreement path before automation.


In [ ]:
# Cross-cutting: simple SLO helper for vision endpoints
from dataclasses import dataclass

@dataclass
class VisionSLO:
    availability: float = 0.995
    p95_ms: int = 4000
    max_critical_field_error_rate: float = 0.005

def breached(slo: VisionSLO, avail: float, p95: int, crit_err: float) -> list[str]:
    out = []
    if avail < slo.availability: out.append("availability")
    if p95 > slo.p95_ms: out.append("latency")
    if crit_err > slo.max_critical_field_error_rate: out.append("critical_accuracy")
    return out or ["ok"]

print("VLM fundamentals", breached(VisionSLO(), 0.99, 5200, 0.02))


### Try it yourself — VLM fundamentals ops

1. Write a one-page runbook section for on-call when VLM fundamentals critical_accuracy SLO breaches.
2. Add a dashboard sketch: cost/1k images, CER/field error, HITL rate, p95 latency.
